# Notebook 01 — Data Loading & Validation

## 1. Objectif du notebook

L'objectif de ce notebook est de charger les données du projet de rééducation
neuromotrice et de vérifier leur qualité avant de commencer l'analyse.

Les principales étapes sont :

1. Charger les différents fichiers CSV.
2. Comprendre la structure des données.
3. Vérifier les types de données.
4. Détecter les valeurs manquantes.
5. Détecter les doublons.
6. Vérifier les clés primaires.
7. Vérifier les relations entre les tables.
8. Vérifier certaines règles métier.
9. Produire un premier rapport de qualité des données.

Ce notebook constitue la première étape du pipeline Data Analysis :

RAW DATA
    ↓
DATA VALIDATION
    ↓
DATA CLEANING
    ↓
DATA INTEGRATION
    ↓
EDA
    ↓
KPI
    ↓
DASHBOARD

In [65]:
import pandas as pd
import numpy as np
import os

In [66]:
DATA_PATH = "../data/raw/"

In [67]:
patients = pd.read_csv(DATA_PATH + "patients.csv")
games = pd.read_csv(DATA_PATH + "games.csv")
exercises = pd.read_csv(DATA_PATH + "exercises.csv")
exercise_levels = pd.read_csv(DATA_PATH + "exercise_levels.csv")
patient_exercises = pd.read_csv(DATA_PATH + "patient_exercises.csv")
level_changes = pd.read_csv(DATA_PATH + "level_changes.csv")
sessions = pd.read_csv(DATA_PATH + "sessions.csv")
results = pd.read_csv(DATA_PATH + "results.csv")

In [68]:
print("Patients :", patients.shape)
print("Games :", games.shape)
print("Exercises :", exercises.shape)
print("Exercise levels :", exercise_levels.shape)
print("Patient exercises :", patient_exercises.shape)
print("Level changes :", level_changes.shape)
print("Sessions :", sessions.shape)
print("Results :", results.shape)

Patients : (20, 7)
Games : (3, 4)
Exercises : (8, 4)
Exercise levels : (24, 7)
Patient exercises : (43, 8)
Level changes : (73, 8)
Sessions : (500, 6)
Results : (500, 5)


In [69]:
datasets = {
    "patients": patients,
    "games": games,
    "exercises": exercises,
    "exercise_levels": exercise_levels,
    "patient_exercises": patient_exercises,
    "level_changes": level_changes,
    "sessions": sessions,
    "results": results
}

for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    display(df.head())


===== PATIENTS =====


,id,first_name,last_name,date_of_birth,patient_code,therapist_id,created_at
0,1,Yasmine,Bouzid,2011-12-18,KK-2026-001,3,2026-04-23 22:12:54
1,2,Amira,Jebali,2018-03-22,KK-2026-002,1,2026-06-17 07:41:35
2,3,Fatma,Tlili,2020-08-26,KK-2026-003,3,2026-05-14 23:54:18
3,4,Malek,Hamdi,2016-11-13,KK-2026-004,2,2026-07-08 08:44:24
4,5,Zied,Gharbi,2015-12-17,KK-2026-005,3,2026-07-06 16:55:18



===== GAMES =====


,id,name,body_part,description
0,1,Bubble Reach,Shoulder,Arcade game where the child pops floating bubb...
1,2,Step Quest,Knee,Adventure game where the child controls a hero...
2,3,Wrist Wizard,Wrist,Magic-themed game where wrist and forearm move...



===== EXERCISES =====


...

,id,patient_exercise_id,from_level_id,to_level_id,reason,session_id,changed_by,changed_at
0,1,1,NaN,20,initial,NaN,3.0,2026-04-26 11:12:54
1,2,2,NaN,23,initial,NaN,3.0,2026-04-30 14:55:42
2,3,3,NaN,22,initial,NaN,3.0,2026-05-20 14:54:18
3,4,4,NaN,2,initial,NaN,2.0,2026-05-24 11:08:54
4,5,5,NaN,13,initial,NaN,3.0,2026-05-24 14:58:28



===== SESSIONS =====


,id,patient_exercise_id,exercise_level_id,session_date,duration,score
0,1,1,20,2026-04-30 15:50:31,4,60.6
1,2,1,20,2026-05-04 15:00:15,8,52.5
2,3,2,23,2026-05-05 15:20:38,6,47.1
3,4,1,20,2026-05-07 15:00:09,5,63.8
4,5,2,23,2026-05-08 15:20:48,7,48.7



===== RESULTS =====


,id,session_id,repetitions,success_rate,progression
0,1,1,9,60.2,NaN
1,2,2,9,52.7,-13.37
2,3,3,10,52.7,NaN
3,4,4,10,68.7,21.52
4,5,5,13,61.3,3.40


In [70]:
for name, df in datasets.items():
    print(
        f"{name:20} → "
        f"{df.shape[0]:4} lignes | "
        f"{df.shape[1]:2} colonnes"
    )

patients             →   20 lignes |  7 colonnes
games                →    3 lignes |  4 colonnes
exercises            →    8 lignes |  4 colonnes
exercise_levels      →   24 lignes |  7 colonnes
patient_exercises    →   43 lignes |  8 colonnes
level_changes        →   73 lignes |  8 colonnes
sessions             →  500 lignes |  6 colonnes
results              →  500 lignes |  5 colonnes


In [71]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


PATIENTS
['id', 'first_name', 'last_name', 'date_of_birth', 'patient_code', 'therapist_id', 'created_at']

GAMES
['id', 'name', 'body_part', 'description']

EXERCISES
['id', 'game_id', 'name', 'description']

EXERCISE_LEVELS
['id', 'exercise_id', 'level_number', 'title', 'target_angle', 'required_score', 'required_progression']

PATIENT_EXERCISES
['id', 'patient_id', 'exercise_id', 'start_level_id', 'current_level_id', 'assigned_by', 'assigned_at', 'is_active']

LEVEL_CHANGES
['id', 'patient_exercise_id', 'from_level_id', 'to_level_id', 'reason', 'session_id', 'changed_by', 'changed_at']

SESSIONS
['id', 'patient_exercise_id', 'exercise_level_id', 'session_date', 'duration', 'score']

RESULTS
['id', 'session_id', 'repetitions', 'success_rate', 'progression']


In [72]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.info())


PATIENTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             20 non-null     int64 
 1   first_name     20 non-null     object
 2   last_name      20 non-null     object
 3   date_of_birth  20 non-null     object
 4   patient_code   20 non-null     object
 5   therapist_id   20 non-null     int64 
 6   created_at     20 non-null     object
dtypes: int64(2), object(5)
memory usage: 1.2+ KB
None

GAMES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3 non-null      int64 
 1   name         3 non-null      object
 2   body_part    3 non-null      object
 3   description  3 non-null      object
dtypes: int64(1), object(3)
memory usage: 228.0+ bytes
None

EXERCISES
<class 'pandas.core.fr

In [73]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print(df.dtypes)


===== PATIENTS =====
id                int64
first_name       object
last_name        object
date_of_birth    object
patient_code     object
therapist_id      int64
created_at       object
dtype: object

===== GAMES =====
id              int64
name           object
body_part      object
description    object
dtype: object

===== EXERCISES =====
id              int64
game_id         int64
name           object
description    object
dtype: object

===== EXERCISE_LEVELS =====
id                        int64
exercise_id               int64
level_number              int64
title                    object
target_angle            float64
required_score          float64
required_progression    float64
dtype: object

===== PATIENT_EXERCISES =====
id                   int64
patient_id           int64
exercise_id          int64
start_level_id       int64
current_level_id     int64
assigned_by          int64
assigned_at         object
is_active             bool
dtype: object

===== LEVEL_CHANGES =

In [74]:
patients["date_of_birth"] = pd.to_datetime(
    patients["date_of_birth"]
)

patients["created_at"] = pd.to_datetime(
    patients["created_at"]
)

In [75]:
patient_exercises["assigned_at"] = pd.to_datetime(
    patient_exercises["assigned_at"]
)

In [76]:
level_changes["changed_at"] = pd.to_datetime(
    level_changes["changed_at"]
)

In [77]:
sessions["session_date"] = pd.to_datetime(
    sessions["session_date"]
)

In [78]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print(df.isnull().sum())


===== PATIENTS =====
id               0
first_name       0
last_name        0
date_of_birth    0
patient_code     0
therapist_id     0
created_at       0
dtype: int64

===== GAMES =====
id             0
name           0
body_part      0
description    0
dtype: int64

===== EXERCISES =====
id             0
game_id        0
name           0
description    0
dtype: int64

===== EXERCISE_LEVELS =====
id                      0
exercise_id             0
level_number            0
title                   0
target_angle            0
required_score          0
required_progression    0
dtype: int64

===== PATIENT_EXERCISES =====
id                  0
patient_id          0
exercise_id         0
start_level_id      0
current_level_id    0
assigned_by         0
assigned_at         0
is_active           0
dtype: int64

===== LEVEL_CHANGES =====
id                      0
patient_exercise_id     0
from_level_id          43
to_level_id             0
reason                  0
session_id             45
c

In [79]:
def missing_values_report(df):
    report = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_percentage": df.isnull().mean() * 100
    })
    
    return report.sort_values(
        "missing_percentage",
        ascending=False
    )

In [80]:
missing_values_report(level_changes)

,missing_count,missing_percentage
session_id,45,61.643836
from_level_id,43,58.904110
changed_by,28,38.356164
id,0,0.000000
patient_exercise_id,0,0.000000
to_level_id,0,0.000000
reason,0,0.000000
changed_at,0,0.000000


In [81]:
for name, df in datasets.items():
    print(
        f"{name:20} → "
        f"{df.duplicated().sum()} doublons"
    )

patients             → 0 doublons
games                → 0 doublons
exercises            → 0 doublons
exercise_levels      → 0 doublons
patient_exercises    → 0 doublons
level_changes        → 0 doublons
sessions             → 0 doublons
results              → 0 doublons


In [82]:
primary_keys = {
    "patients": ("id", patients),
    "games": ("id", games),
    "exercises": ("id", exercises),
    "exercise_levels": ("id", exercise_levels),
    "patient_exercises": ("id", patient_exercises),
    "level_changes": ("id", level_changes),
    "sessions": ("id", sessions),
    "results": ("id", results)  
}

for name, (pk, df) in primary_keys.items():
    duplicates = df[pk].duplicated().sum()
    
    print(
        f"{name:20} | PK = {pk:5} | "
        f"Doublons = {duplicates}"
    )

patients             | PK = id    | Doublons = 0
games                | PK = id    | Doublons = 0
exercises            | PK = id    | Doublons = 0
exercise_levels      | PK = id    | Doublons = 0
patient_exercises    | PK = id    | Doublons = 0
level_changes        | PK = id    | Doublons = 0
sessions             | PK = id    | Doublons = 0
results              | PK = id    | Doublons = 0


In [83]:

for name, (pk, df) in primary_keys.items():
    nulls = df[pk].isnull().sum()
    
    print(
        f"{name:20} | PK = {pk:5} | "
        f"NULL = {nulls}"
    )

patients             | PK = id    | NULL = 0
games                | PK = id    | NULL = 0
exercises            | PK = id    | NULL = 0
exercise_levels      | PK = id    | NULL = 0
patient_exercises    | PK = id    | NULL = 0
level_changes        | PK = id    | NULL = 0
sessions             | PK = id    | NULL = 0
results              | PK = id    | NULL = 0


In [84]:
invalid_patient_ids = patient_exercises[
    ~patient_exercises["patient_id"].isin(patients["id"])
]

invalid_patient_ids

,id,patient_id,exercise_id,start_level_id,current_level_id,assigned_by,assigned_at,is_active


In [85]:
invalid_game_ids = exercises[
    ~exercises["game_id"].isin(games["id"])
]

invalid_game_ids

,id,game_id,name,description


In [86]:
invalid_exercise_ids = exercise_levels[
    ~exercise_levels["exercise_id"].isin(exercises["id"])
]

invalid_exercise_ids

,id,exercise_id,level_number,title,target_angle,required_score,required_progression


In [87]:
invalid_patient_ids = patient_exercises[
    ~patient_exercises["patient_id"].isin(patients["id"])
]

invalid_patient_ids

,id,patient_id,exercise_id,start_level_id,current_level_id,assigned_by,assigned_at,is_active


In [88]:
invalid_start_levels = patient_exercises[
    ~patient_exercises["start_level_id"].isin(exercise_levels["id"])
]

invalid_current_levels = patient_exercises[
    ~patient_exercises["current_level_id"].isin(exercise_levels["id"])
]

In [89]:
invalid_patient_exercises = level_changes[
    ~level_changes["patient_exercise_id"].isin(
        patient_exercises["id"]
    )
]

invalid_patient_exercises

,id,patient_exercise_id,from_level_id,to_level_id,reason,session_id,changed_by,changed_at


In [90]:
invalid_session_patient_exercises = sessions[
    ~sessions["patient_exercise_id"].isin(
        patient_exercises["id"]
    )
]

invalid_session_patient_exercises

,id,patient_exercise_id,exercise_level_id,session_date,duration,score


In [91]:
invalid_session_levels = sessions[
    ~sessions["exercise_level_id"].isin(
        exercise_levels["id"]
    )
]

invalid_session_levels

,id,patient_exercise_id,exercise_level_id,session_date,duration,score


In [92]:
invalid_result_sessions = results[
    ~results["session_id"].isin(
        sessions["id"]
    )
]

invalid_result_sessions

,id,session_id,repetitions,success_rate,progression


In [93]:
for exercise_id, group in exercise_levels.groupby("exercise_id"):
    
    levels = sorted(group["level_number"].tolist())
    
    print(
        f"Exercise {exercise_id} → levels = {levels}"
    )

Exercise 1 → levels = [1, 2, 3]
Exercise 2 → levels = [1, 2, 3]
Exercise 3 → levels = [1, 2, 3]
Exercise 4 → levels = [1, 2, 3]
Exercise 5 → levels = [1, 2, 3]
Exercise 6 → levels = [1, 2, 3]
Exercise 7 → levels = [1, 2, 3]
Exercise 8 → levels = [1, 2, 3]


In [94]:
pe_check = patient_exercises.merge(
    exercise_levels[["id", "exercise_id"]],
    left_on="start_level_id",
    right_on="id",
    suffixes=("", "_level")
)

pe_check[
    pe_check["exercise_id"] != pe_check["exercise_id_level"]
]

,id,patient_id,exercise_id,start_level_id,current_level_id,assigned_by,assigned_at,is_active,id_level,exercise_id_level


In [95]:
pe_check = patient_exercises.merge(
    exercise_levels[["id", "exercise_id"]],
    left_on="current_level_id",
    right_on="id",
    suffixes=("", "_level")
)

pe_check[
    pe_check["exercise_id"] != pe_check["exercise_id_level"]
]

,id,patient_id,exercise_id,start_level_id,current_level_id,assigned_by,assigned_at,is_active,id_level,exercise_id_level


In [96]:
level_changes[
    level_changes["reason"] == "initial"
][
    ["from_level_id", "to_level_id", "reason"]
].head()

,from_level_id,to_level_id,reason
0,NaN,20,initial
1,NaN,23,initial
2,NaN,22,initial
3,NaN,2,initial
4,NaN,13,initial


In [97]:
level_changes["reason"].value_counts()

reason
initial             43
auto_progression    17
auto_score          11
manual               2
Name: count, dtype: int64

In [98]:
invalid_scores = sessions[
    (sessions["score"] < 0) |
    (sessions["score"] > 100)
]

print(
    "Nombre de scores invalides :",
    len(invalid_scores)
)

invalid_scores

Nombre de scores invalides : 0


,id,patient_exercise_id,exercise_level_id,session_date,duration,score


In [99]:
invalid_duration = sessions[
    sessions["duration"] <= 0
]

print(
    "Nombre de durées invalides :",
    len(invalid_duration)
)

invalid_duration

Nombre de durées invalides : 0


,id,patient_exercise_id,exercise_level_id,session_date,duration,score


In [100]:
invalid_success_rate = results[
    (results["success_rate"] < 0) |
    (results["success_rate"] > 100)
]

print(
    "Nombre de success_rate invalides :",
    len(invalid_success_rate)
)

invalid_success_rate

Nombre de success_rate invalides : 0


,id,session_id,repetitions,success_rate,progression


In [101]:
invalid_repetitions = results[
    results["repetitions"] < 0
]

print(
    "Nombre de répétitions invalides :",
    len(invalid_repetitions)
)

invalid_repetitions

Nombre de répétitions invalides : 0


,id,session_id,repetitions,success_rate,progression


In [102]:
duplicate_results = results[
    results["session_id"].duplicated(keep=False)
]

duplicate_results

,id,session_id,repetitions,success_rate,progression


In [103]:
duplicate_results["session_id"].nunique()

0

In [104]:
sessions_without_results = sessions[
    ~sessions["id"].isin(results["session_id"])
]

print(
    "Sessions sans résultat :",
    len(sessions_without_results)
)

sessions_without_results

Sessions sans résultat : 0


,id,patient_exercise_id,exercise_level_id,session_date,duration,score


In [105]:
results_without_session = results[
    ~results["session_id"].isin(sessions["id"])
]

print(
    "Résultats sans session :",
    len(results_without_session)
)

results_without_session

Résultats sans session : 0


,id,session_id,repetitions,success_rate,progression


In [106]:
session_dates = sessions.merge(
    patient_exercises[
        ["id", "assigned_at"]
    ],
    left_on="patient_exercise_id",
    right_on="id",
    suffixes=("_session", "_assignment")
)

invalid_chronology = session_dates[
    session_dates["session_date"] <
    session_dates["assigned_at"]
]

print(
    "Sessions avant l'affectation :",
    len(invalid_chronology)
)

invalid_chronology

Sessions avant l'affectation : 0


,id_session,patient_exercise_id,exercise_level_id,session_date,duration,score,id_assignment,assigned_at


In [107]:
print("SESSIONS")
display(sessions.describe())

print("RESULTS")
display(results.describe())

SESSIONS


,id,patient_exercise_id,exercise_level_id,session_date,duration,score
count,500.000000,500.000000,500.000000,500,500.000000,500.000000
mean,250.500000,20.806000,14.800000,2026-08-02 14:23:15.534000128,7.842000,56.777800
min,1.000000,1.000000,1.000000,2026-04-30 15:50:31,3.000000,24.200000
25%,125.750000,11.000000,11.000000,2026-07-13 02:04:07,7.000000,51.200000
50%,250.500000,21.000000,17.000000,2026-08-08 10:00:07,8.000000,56.750000
75%,375.250000,30.000000,20.000000,2026-08-28 15:35:18.249999872,9.000000,62.225000
max,500.000000,43.000000,23.000000,2026-09-22 16:30:42,14.000000,85.100000
std,144.481833,11.668619,6.863397,NaN,1.914253,8.465583


RESULTS


,id,session_id,repetitions,success_rate,progression
count,500.000000,500.000000,500.000000,500.000000,457.000000
mean,250.500000,250.500000,14.164000,57.340800,3.003961
std,144.481833,144.481833,3.490196,8.826078,15.304562
min,1.000000,1.000000,5.000000,24.600000,-49.860000
25%,125.750000,125.750000,12.000000,51.775000,-5.110000
50%,250.500000,250.500000,14.000000,57.250000,1.760000
75%,375.250000,375.250000,16.000000,63.100000,10.150000
max,500.000000,500.000000,26.000000,82.600000,107.020000


In [108]:
results["progression"].describe()

count    457.000000
mean       3.003961
std       15.304562
min      -49.860000
25%       -5.110000
50%        1.760000
75%       10.150000
max      107.020000
Name: progression, dtype: float64

In [109]:
progression_outliers = results[
    results["progression"].notna() &
    (
        (results["progression"] > 100) |
        (results["progression"] < -100)
    )
]

print(
    "Progressions très élevées ou faibles :",
    len(progression_outliers)
)

progression_outliers

Progressions très élevées ou faibles : 1


,id,session_id,repetitions,success_rate,progression
244,245,245,17,54.9,107.02


In [ ]:
quality_report = []

for name, df in datasets.items():
    quality_report.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicates": df.duplicated().sum(),
        "missing_values": df.isnull().sum().sum()
    })

quality_report = pd.DataFrame(quality_report)

quality_report

,table,rows,columns,duplicates,missing_values
0,patients,20,7,0,0
1,games,3,4,0,0
2,exercises,8,4,0,0
3,exercise_levels,24,7,0,0
4,patient_exercises,43,8,0,0
5,level_changes,73,8,0,116
6,sessions,500,6,0,0
7,results,500,5,0,43


: 

# Conclusion

Le Notebook 01 a permis de charger et de valider les données du projet.

Nous avons vérifié :

- la structure des tables ;
- les types de données ;
- les valeurs manquantes ;
- les doublons ;
- les clés primaires ;
- les clés étrangères ;
- les niveaux des exercices ;
- les affectations des patients ;
- les changements de niveau ;
- les sessions ;
- les résultats ;
- les scores ;
- les taux de réussite ;
- les répétitions ;
- les durées ;
- la chronologie des sessions ;
- la progression.

Les anomalies éventuellement détectées seront traitées dans le Notebook 02 :
`02_data_cleaning.ipynb`.